# SkillLens AI — Preprocessing and Feature Engineering

**Stage 3 of the SkillLens AI project.**

This notebook builds a clean, leakage-safe preprocessing pipeline: split the data into training and testing sets first, then fit scaling only on the training set. This exact order matters — it's what later stages (K-Means, DBSCAN, PCA, classification) will rely on.

**Important:** `generation_reference.csv` is intentionally **not** loaded anywhere in this notebook, for the same reason as in the EDA notebook — it must never influence feature analysis or modeling.

No target/label column, K-Means clustering, DBSCAN, PCA, or classifier is created in this notebook. This stage is preprocessing only.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Load the Dataset

We load only `student_data.csv`. `generation_reference.csv` is never read in this notebook.

In [2]:
data = pd.read_csv("../data/raw/student_data.csv")
print("Dataset loaded:", data.shape)

Dataset loaded: (300, 10)


## 3. Select Features (and Exclude `student_id`)

We use the same 9 features established during EDA. `student_id` is excluded from the feature list.

**Why `student_id` must not be used as a model feature:** it is just an arbitrary, sequential number assigned when the dataset was generated — it has no real relationship to a student's skill level. If a model were allowed to use it, it could end up "learning" spurious patterns tied to row order or ID number instead of genuine skill patterns. It also would not generalize: a brand-new student has no meaningful ID to compare against. So `student_id` is kept only as a row identifier, never as an input to any model.

In [3]:
features = [
    "cgpa", "coding_skill", "dsa_skill", "math_aptitude", "communication_skill",
    "security_knowledge", "projects_count", "internships_count", "certifications_count",
]

X = data[features]
print("Feature matrix shape:", X.shape)
print("student_id excluded. Columns used as features:", X.columns.tolist())

Feature matrix shape: (300, 9)
student_id excluded. Columns used as features: ['cgpa', 'coding_skill', 'dsa_skill', 'math_aptitude', 'communication_skill', 'security_knowledge', 'projects_count', 'internships_count', 'certifications_count']


## 4. Original Feature Ranges (Before Any Processing)

Recorded here so we can later compare against the scaled values and see exactly what changed.

In [4]:
original_ranges = pd.DataFrame({
    "min": X.min(),
    "max": X.max(),
    "mean": X.mean().round(2),
    "std": X.std().round(2),
})
original_ranges

,min,max,mean,std
cgpa,2.8,10.0,6.97,1.42
coding_skill,0.1,10.0,6.56,2.22
dsa_skill,0.1,10.0,6.18,2.35
math_aptitude,0.5,10.0,6.26,1.92
communication_skill,0.9,10.0,5.46,1.78
security_knowledge,0.0,10.0,5.63,2.07
projects_count,0.0,9.0,4.03,2.14
internships_count,0.0,4.0,1.15,1.13
certifications_count,0.0,5.0,1.96,1.55


## 5. Train/Test Split (Done BEFORE Any Scaling)

We split the data into 80% training and 20% testing **before** fitting any scaler. This order is deliberate: if we scaled the entire dataset first and split afterward, information from the test set (its mean and spread) would leak into the training process through the scaler — the model would then be evaluated in a way that's slightly too optimistic, since the scaler would have "seen" the test data's statistics in advance. Splitting first avoids this entirely.

In [5]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Testing set shape: ", X_test.shape)

Training set shape: (240, 9)
Testing set shape:  (60, 9)


## 6. Fit StandardScaler on Training Data Only

`StandardScaler` transforms each feature so it has mean 0 and standard deviation 1: `scaled_value = (value - mean) / standard_deviation`.

**We call `.fit()` only on `X_train`.** This means the mean and standard deviation used for scaling come only from the training data. We then use that same already-fitted scaler to `.transform()` both `X_train` and `X_test` — the test set's own statistics are never used to fit anything. This is what keeps the workflow leakage-safe.

In [6]:
scaler = StandardScaler()

# Fit ONLY on the training data - the scaler must never see the test data's statistics.
scaler.fit(X_train)

# Now use the already-fitted scaler to transform both sets.
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=features, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features, index=X_test.index)

print("Scaling complete.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape: ", X_test_scaled.shape)

Scaling complete.
X_train_scaled shape: (240, 9)
X_test_scaled shape:  (60, 9)


## 7. Scaler Means and Standard Deviations

These are the exact numbers the scaler learned from `X_train` — the values it will use (via `.transform()`) on any new data, including the test set and, later, a real student's submitted profile.

In [7]:
scaler_stats = pd.DataFrame({
    "mean_learned_from_train": scaler.mean_,
    "std_learned_from_train": scaler.scale_,
}, index=features)
scaler_stats

,mean_learned_from_train,std_learned_from_train
cgpa,7.013750,1.371897
coding_skill,6.582083,2.181774
dsa_skill,6.186250,2.367457
math_aptitude,6.250417,1.889599
communication_skill,5.434583,1.797131
security_knowledge,5.666667,2.093714
projects_count,4.066667,2.135936
internships_count,1.145833,1.128967
certifications_count,2.016667,1.557152


## 8. Before/After Scaling Example

A small side-by-side look at the first 5 training rows, before and after scaling, for one easy-to-follow feature (`coding_skill`) plus the full row comparison.

In [8]:
comparison_example = pd.DataFrame({
    "coding_skill_before": X_train["coding_skill"].head(5),
    "coding_skill_after": X_train_scaled["coding_skill"].head(5),
})
comparison_example

,coding_skill_before,coding_skill_after
232,8.0,0.649892
59,6.9,0.145715
6,9.7,1.429074
185,2.3,-1.962661
173,6.7,0.054046


In [9]:
print("Full first row - before scaling:")
print(X_train.iloc[0])
print()
print("Full first row - after scaling:")
print(X_train_scaled.iloc[0])

Full first row - before scaling:
cgpa                    7.1
coding_skill            8.0
dsa_skill               9.0
math_aptitude           8.2
communication_skill     3.5
security_knowledge      6.4
projects_count          6.0
internships_count       3.0
certifications_count    4.0
Name: 232, dtype: float64

Full first row - after scaling:
cgpa                    0.062869
coding_skill            0.649892
dsa_skill               1.188512
math_aptitude           1.031745
communication_skill    -1.076484
security_knowledge      0.350255
projects_count          0.905146
internships_count       1.642357
certifications_count    1.273693
Name: 232, dtype: float64


## 9. Confirm Scaled Training Features Have Mean ≈ 0 and Std ≈ 1

This is the standard sanity check for `StandardScaler`: the training data it was fit on should now average out to mean 0 and standard deviation 1 for every feature. (The test set, scaled using the training statistics, will be *close* to but not exactly 0/1 — that's expected and checked separately below.)

In [10]:
print("X_train_scaled mean per feature (should be ~0):")
print(X_train_scaled.mean().round(3))
print()
print("X_train_scaled standard deviation per feature (should be ~1):")
print(X_train_scaled.std(ddof=0).round(3))

print()
print("For comparison, X_test_scaled mean per feature (expected close to, but not exactly, 0):")
print(X_test_scaled.mean().round(3))
print()
print("X_test_scaled standard deviation per feature (expected close to, but not exactly, 1):")
print(X_test_scaled.std(ddof=0).round(3))

X_train_scaled mean per feature (should be ~0):
cgpa                    0.0
coding_skill            0.0
dsa_skill              -0.0
math_aptitude           0.0
communication_skill    -0.0
security_knowledge     -0.0
projects_count          0.0
internships_count       0.0
certifications_count    0.0
dtype: float64

X_train_scaled standard deviation per feature (should be ~1):
cgpa                    1.0
coding_skill            1.0
dsa_skill               1.0
math_aptitude           1.0
communication_skill     1.0
security_knowledge      1.0
projects_count          1.0
internships_count       1.0
certifications_count    1.0
dtype: float64

For comparison, X_test_scaled mean per feature (expected close to, but not exactly, 0):
cgpa                   -0.153
coding_skill           -0.052
dsa_skill              -0.004
math_aptitude           0.031
communication_skill     0.061
security_knowledge     -0.087
projects_count         -0.094
internships_count       0.004
certifications_count   -0.

## 10. Why StandardScaler Suits K-Means, DBSCAN, and PCA

- **K-Means** groups points by measuring the *distance* between them and cluster centers. If features are left on their original scales, a feature like `security_knowledge` (range 0–10) would influence that distance far more than `internships_count` (range 0–4) purely because its numbers are bigger — not because it's a more important skill. Scaling puts every feature on equal footing.
- **DBSCAN** also relies entirely on distance (it defines "dense regions" using a distance threshold, `eps`). The same scale problem applies: without scaling, one large-range feature could dominate which points count as "close" to each other.
- **PCA** finds directions of maximum variance in the data. A feature with a naturally larger numeric range will show more raw variance, even if it isn't actually more informative — PCA could then be misled into treating it as the most important direction just because of its units, not its real signal. Scaling first ensures PCA reflects genuine patterns rather than measurement scale.

In short: all three algorithms are scale-sensitive, and our features are not naturally on the same scale (see Section 4), so standardization is a necessary step before any of them are used.

## 11. Outliers Are Not Removed

Stage 2 (EDA) found only 3 outliers total across all 300 students, using the IQR method: 2 in `cgpa` and 1 in `math_aptitude`. All 7 other features had 0 outliers.

These values are not invalid — they still fall well within each feature's valid domain range (e.g., no `cgpa` value below 0 or above 10). They simply represent students who are unusually strong or weak compared to the rest of the sample, which is a realistic thing to see in any group of students. Given how few there are (3 out of 300 rows) and that they are legitimate, in-range values, we do not remove or cap them. Deleting real data points just to make later clustering look cleaner would be a form of manipulating the results, which goes against this project's principle of not fabricating outcomes.

We also do not create any new engineered features or perform feature selection at this stage — the original 9 features are kept as-is, since there is no clear, justified mathematical reason yet to add or remove any.

## 12. Save the Fitted Scaler

We save the scaler fitted on `X_train` so later stages (and eventually the FastAPI backend) can reuse the exact same scaling — a new student's raw skill values must be transformed with these same training-derived mean/std values, not re-fit on new data.

In [11]:
joblib.dump(scaler, "../models/scaler.pkl")
print("Saved fitted scaler to ml/models/scaler.pkl")

Saved fitted scaler to ml/models/scaler.pkl


## Preprocessing Conclusion

- **`student_id` was excluded** because it is an arbitrary, sequential identifier with no genuine relationship to a student's skills. Using it as a feature risks learning meaningless patterns tied to row order, and it wouldn't generalize to new students.
- **Train/test splitting happened first, before any scaling,** so that no transformation applied to the training data could be influenced by information from the test data. This is the core rule that keeps the whole workflow leakage-safe.
- **`StandardScaler` was used** because our 9 features are on very different numeric scales (skills roughly 0–10, counts roughly 0–5), and the algorithms planned for later stages (K-Means, DBSCAN, PCA) all measure distance or variance in ways that are sensitive to scale. Standardizing puts every feature on equal footing.
- **Fitting the scaler only on training data** means its mean and standard deviation come exclusively from `X_train`. The test set is transformed using those same numbers, but never contributes to learning them — this mirrors how a real, brand-new student's data would be handled in production: transformed using statistics learned in advance, never used to redefine them.
- **What Stage 4 will do next:** using `X_train_scaled` only, run K-Means to discover natural groupings among students, interpret those groups, and assign each training student a `skill_category` label. The already-fitted K-Means model will then be used (not refit) to assign cluster labels to `X_test_scaled`, so the test set's evaluation labels come from a model that never trained on the test data either.

No target/label, K-Means, DBSCAN, PCA, or classifier was created in this notebook.